# Re-prompt API spike
Run top to bottom on a GPU runtime. Adds a box at a middle frame t of three DAVIS val videos and resumes propagation in three ways, then compares the masks with the run that had no re-prompt. Cells: mount Drive, set the data root, get the repo, install, download checkpoints, run Tiny and Large, show the CSV.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
DATA_ROOT = ""  # @param {type:"string"}
assert DATA_ROOT, "Fill in DATA_ROOT (the Drive folder that holds the datasets)"
assert os.path.exists(DATA_ROOT), f"DATA_ROOT does not exist: {DATA_ROOT}"
os.environ["SAM2FR_DATA_ROOT"] = DATA_ROOT

In [ ]:
# After this phase is merged, set BRANCH to "main".
BRANCH = "phase-1a-data"  # @param {type:"string"}
REPO_URL = "https://github.com/VEDITHREDDY26/sam2-failure-recovery.git"
REPO_DIR = "/content/sam2-failure-recovery"
import os
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} fetch origin
else:
    !git clone {REPO_URL} {REPO_DIR}
!git -C {REPO_DIR} checkout {BRANCH}
!git -C {REPO_DIR} pull origin {BRANCH}
%cd {REPO_DIR}

In [ ]:
# Colab ships plain, headless and contrib OpenCV builds that share one cv2 folder;
# remove all three, then install only the pinned contrib build.
!pip uninstall -y -q opencv-python opencv-python-headless opencv-contrib-python
!pip install -q -r requirements-colab.txt
!pip install -q --force-reinstall --no-deps opencv-contrib-python==4.11.0.86
!pip list 2>/dev/null | grep -i opencv

Restart the session now (Runtime > Restart session), then re-run the Drive, DATA_ROOT and repo cells and continue from the check cell below; skip the install cell. If `cv2` has no version or `TrackerCSRT_create` prints False, the install cell did not finish: run it again, then restart.

In [ ]:
import platform, numpy, cv2, torch
print("python", platform.python_version())
print("torch", torch.__version__)
print("cuda available", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu", torch.cuda.get_device_name(0))
print("numpy", numpy.__version__)
print("cv2", cv2.__version__)
print("TrackerCSRT_create", hasattr(cv2, "TrackerCSRT_create"))

In [ ]:
!bash scripts/get_checkpoints.sh

In [ ]:
!python -m experiments.spikes.reprompt --model tiny

In [ ]:
!python -m experiments.spikes.reprompt --model large

In [ ]:
import pandas as pd, sys
sys.path.insert(0, ".")
from src.sam2_runner.smoke import load_data_config
from experiments.spikes.reprompt import output_path

cfg = load_data_config()
for model in ("tiny", "large"):
    path = output_path(cfg, model)
    print(path)
    print(pd.read_csv(path).to_string(index=False))